# Contrastive Direction Recipe

Representation-engineering workflows often start with paired positive and negative examples, compute an activation-space direction, then steer a neutral or target example along that direction. This notebook demonstrates that pattern with a tiny local classifier and TorchLens replay hooks.


In [ ]:
from __future__ import annotations

import torch
from torch import nn

import torchlens as tl


class TinySentimentModel(nn.Module):
    """Tiny model with a readable activation direction."""

    def __init__(self) -> None:
        """Initialize deterministic encoder and two-logit head."""

        super().__init__()
        self.encoder = nn.Linear(6, 6, bias=False)
        self.head = nn.Linear(6, 2, bias=False)
        with torch.no_grad():
            self.encoder.weight.copy_(torch.eye(6))
            self.head.weight.zero_()
            self.head.weight[0, 0] = -1.0
            self.head.weight[1, 0] = 1.0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Classify examples after a ReLU feature map.

        Parameters
        ----------
        x:
            Batch of six-dimensional toy features.

        Returns
        -------
        torch.Tensor
            Two class logits.
        """

        return self.head(torch.relu(self.encoder(x)))


model = TinySentimentModel().eval()
positive_examples = torch.tensor([[2.0, 1.0, 0.0, 0.0, 0.0, 0.0], [1.5, 1.0, 0.0, 0.0, 0.0, 0.0]])
negative_examples = torch.tensor([[-2.0, 0.0, 0.0, 1.0, 0.0, 0.0], [-1.5, 0.0, 0.0, 1.0, 0.0, 0.0]])
neutral_example = torch.tensor([[0.1, 1.0, 0.0, 0.0, 0.0, 0.0]])

Capture positive and negative batches at the same site. The direction is just the mean positive activation minus the mean negative activation.


In [ ]:
positive_log = tl.log_forward_pass(
    model, positive_examples, vis_opt="none", intervention_ready=True
)
negative_log = tl.log_forward_pass(
    model, negative_examples, vis_opt="none", intervention_ready=True
)

positive_activation = positive_log.find_sites(tl.func("relu")).first().activation.detach()
negative_activation = negative_log.find_sites(tl.func("relu")).first().activation.detach()
direction = positive_activation.mean(dim=0) - negative_activation.mean(dim=0)
print(direction)
assert direction.shape == (6,)

Now capture a neutral example and replay it with `tl.steer`. `feature_axis=-1` is explicit because vector directions should state which axis is the representation axis.


In [ ]:
base_log = tl.log_forward_pass(model, neutral_example, vis_opt="none", intervention_ready=True)
steered_log = base_log.fork("positive_steering")
steered_log.attach_hooks(
    tl.func("relu"),
    tl.steer(direction, magnitude=0.5, feature_axis=-1),
).replay()

base_logits = base_log.layer_list[-1].activation.detach()
steered_logits = steered_log.layer_list[-1].activation.detach()
print("base logits", base_logits)
print("steered logits", steered_logits)
assert steered_logits[0, 1] > base_logits[0, 1]